In [1]:
import pandas as pd
from pathlib import Path

In [2]:
#Function to combine data by file type
def combineDataFiles(folder, files, output_folder,
                     start_year=2023, end_year=2026):

    output_folder = Path(output_folder)
    output_folder.mkdir(parents=True, exist_ok=True)

    for file in files:

        dfs = []

        for year in range(start_year, end_year + 1):

            year_folder = Path(folder) / str(year)

            matching_files = year_folder.glob(
                f"{file}_data_*.parquet"
            )

            for file_path in matching_files:

                print(f"Loading: {file_path}")

                df = pd.read_parquet(file_path)

                df["year"] = year

                dfs.append(df)

        if dfs:

            combined_df = pd.concat(
                dfs,
                ignore_index=True
            )


            combined_df = combined_df.sort_values(
                "interval_end_utc"
            )

            output_path = (
                output_folder /
                f"{file}_data_{start_year}_{end_year}.parquet"
            )

            combined_df.to_parquet(
                output_path,
                index=False
            )

            print(
                f"Saved {len(combined_df):,} rows to {output_path}"
            )

        else:
            print(f"No files found for {file}")

In [3]:
#Different Files
files = [
    "dam",
    "load",
    "outage",
    "rtm",
    "solar",
    "wind"
]



In [4]:
#Combining the data
combineDataFiles(
    "data-pull/raw_data",
    files,
    "data-pull/combined_data"
)

Loading: data-pull/raw_data/2023/dam_data_2023-01-01_2024-01-01.parquet
Loading: data-pull/raw_data/2024/dam_data_2024-01-01_2025-01-01.parquet
Loading: data-pull/raw_data/2025/dam_data_2025-01-01_2026-01-01.parquet
Loading: data-pull/raw_data/2026/dam_data_2026-01-01_2026-09-17.parquet
Saved 32,520 rows to data-pull/combined_data/dam_data_2023_2026.parquet
Loading: data-pull/raw_data/2023/load_data_2023-01-01_2024-01-01.parquet
Loading: data-pull/raw_data/2024/load_data_2024-01-01_2025-01-01.parquet
Loading: data-pull/raw_data/2025/load_data_2025-01-01_2026-01-01.parquet
Loading: data-pull/raw_data/2026/load_data_2026-01-01_2026-09-17.parquet
Saved 390,240 rows to data-pull/combined_data/load_data_2023_2026.parquet
Loading: data-pull/raw_data/2023/outage_data_2023-01-01_2024-01-01.parquet
Loading: data-pull/raw_data/2024/outage_data_2024-01-01_2025-01-01.parquet
Loading: data-pull/raw_data/2025/outage_data_2025-01-01_2026-01-01.parquet
Loading: data-pull/raw_data/2026/outage_data_2026

In [5]:
#Loading Data to a combined file
dam = pd.read_parquet("data-pull/combined_data/dam_data_2023_2026.parquet")
load = pd.read_parquet("data-pull/combined_data/load_data_2023_2026.parquet")
outage = pd.read_parquet("data-pull/combined_data/outage_data_2023_2026.parquet")
rtm = pd.read_parquet("data-pull/combined_data/rtm_data_2023_2026.parquet")
solar = pd.read_parquet("data-pull/combined_data/solar_data_2023_2026.parquet")
wind = pd.read_parquet("data-pull/combined_data/wind_data_2023_2026.parquet")

In [6]:
#Removin columns
dam = dam.drop(columns=['location', 'location_type'])
rtm = rtm.drop(columns = ['location', 'location_type'])

In [7]:
#filtering our irrelevent projected weather data
# WIND
wind["time_before"] = (
    wind["interval_start_utc"] - wind["publish_time_utc"]
)

wind_before = wind[
    wind["publish_time_utc"] < wind["interval_start_utc"]
].copy()

wind_before["distance_from_24"] = (
    wind_before["time_before"] - pd.Timedelta(hours=24)
).abs()

wind_24 = wind_before.loc[
    wind_before.groupby(
        ["interval_start_utc", "interval_end_utc"]
    )["distance_from_24"].idxmin()
].copy()

# SOLAR
solar["time_before"] = (
    solar["interval_start_utc"] - solar["publish_time_utc"]
)

solar_before = solar[
    solar["publish_time_utc"] < solar["interval_start_utc"]
].copy()

solar_before["distance_from_24"] = (
    solar_before["time_before"] - pd.Timedelta(hours=24)
).abs()

solar_24 = solar_before.loc[
    solar_before.groupby(
        ["interval_start_utc", "interval_end_utc"]
    )["distance_from_24"].idxmin()
].copy()

In [8]:
del solar_before, wind_before,

In [9]:
#joining wind and solar tables
weather = pd.merge(
    wind_24,
    solar_24,
    on = ["interval_start_utc", "interval_end_utc"],
    how = "inner",
    suffixes=("_wind", "_solar")
).sort_values("interval_start_utc")


In [10]:
# Keep everything from this time onward
start_time = pd.Timestamp("2023-03-25 02:00:00+00:00")
rtm = rtm[
    rtm["interval_end_utc"] >= start_time
].copy()
dam = dam[
    dam["interval_end_utc"] >= start_time
].copy()
load = load[
    load["interval_end_utc"] >= start_time
].copy()
outage = outage[
    outage["interval_end_utc"] >= start_time
].copy()
weather = weather[
    weather["interval_end_utc"] >= start_time
].copy()




In [11]:
#checking publish times are equal in wiind and solar
weather["publish_time_utc_solar"].eq(weather["publish_time_utc_wind"]).all()

np.True_

In [12]:
#droping irrelevent columns
weather = weather.drop(columns=["year_wind", 
                                "time_before_wind", 
                                "distance_from_24_wind",
                                "publish_time_utc_solar",
                                "year_solar",
                                "time_before_solar",
                                "distance_from_24_solar"
                                ]).rename(columns = {"publish_time_utc_wind": "publish_time_utc"})

In [15]:
#Moving cleaned data into its own folder/files
rtm.to_parquet("data-pull/cleaned_data/rtm.parquet")
dam.to_parquet("data-pull/cleaned_data/dam.parquet")
load.to_parquet("data-pull/cleaned_data/load.parquet")
outage.to_parquet("data-pull/cleaned_data/outage.parquet")
weather.to_parquet("data-pull/cleaned_data/weather.parquet")
